# Final Canonical Results Assembly

This notebook performs the final mechanical reconciliation before manuscript editing.

**Decision already established by the independent audit:** Stage A2 MNB is canonical because it reproduced the independent refit with 0 disagreements. Stage B MNB is therefore replaced, while all non-MNB Stage B results are retained.

Outputs:
- canonical six-model seed-level results
- canonical six-model summary
- canonical MNB PR-AUC and predicted-positive rate
- manuscript-ready compact performance table
- manuscript-ready diagnostic summary
- revised Macro-F1 figure including label-only reference baselines

In [ ]:
!pip -q install scikit-learn
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, precision_score,
    recall_score, roc_auc_score, average_precision_score, confusion_matrix
)

REV=Path("/content/drive/MyDrive/stress_detection_reviewer_revision")
A2=REV/"stageA2"
B=REV/"stageB"
BR=B/"results"
OUT=REV/"final_canonical"
OUT.mkdir(parents=True,exist_ok=True)

## 1. Recompute complete canonical MNB metrics, including PR-AUC

In [ ]:
datasets=["Dreaddit_Test","Reddit_Title","Reddit_Combi","Twitter_Full","Twitter_NonAdvert"]
mnb_rows=[]

for dname in datasets:
    p=pd.read_csv(A2/f"MNB_CANONICAL_{dname}.csv")
    y=p.true_label.to_numpy()
    pred=p.predicted_label.to_numpy()
    score=p.p_pos.to_numpy()
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()

    mnb_rows.append({
        "model":"Multinomial_NB",
        "seed":np.nan,
        "dataset":dname,
        "accuracy":accuracy_score(y,pred),
        "macro_f1":f1_score(y,pred,average="macro",zero_division=0),
        "balanced_accuracy":balanced_accuracy_score(y,pred),
        "macro_precision":precision_score(y,pred,average="macro",zero_division=0),
        "macro_recall":recall_score(y,pred,average="macro",zero_division=0),
        "positive_recall":recall_score(y,pred,pos_label=1,zero_division=0),
        "predicted_positive_rate":float(np.mean(pred==1)),
        "roc_auc":roc_auc_score(y,score),
        "pr_auc":average_precision_score(y,score),
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)
    })

mnb=pd.DataFrame(mnb_rows)
display(mnb.round(6))
mnb.to_csv(OUT/"CANONICAL_MNB_complete_metrics.csv",index=False)

## 2. Replace stale Stage-B MNB rows with canonical MNB

In [ ]:
stageb_seed=pd.read_csv(BR/"ALL_SIX_MODELS_seedlevel.csv")
non_mnb=stageb_seed[stageb_seed.model!="Multinomial_NB"].copy()

canonical_seed=pd.concat([non_mnb,mnb],ignore_index=True)
canonical_seed.to_csv(OUT/"CANONICAL_ALL_SIX_MODELS_seedlevel.csv",index=False)

metric_cols=["accuracy","macro_f1","balanced_accuracy","positive_recall",
             "predicted_positive_rate","roc_auc","pr_auc"]

rows=[]
for (model,dataset),g in canonical_seed.groupby(["model","dataset"]):
    r={"model":model,"dataset":dataset,"runs":len(g)}
    for c in metric_cols:
        r[f"{c}_mean"]=g[c].mean()
        r[f"{c}_sd"]=g[c].std(ddof=1) if len(g)>1 else np.nan
    rows.append(r)

summary=pd.DataFrame(rows)
summary.to_csv(OUT/"CANONICAL_ALL_SIX_MODELS_summary.csv",index=False)

display(summary[[
    "model","dataset","runs","macro_f1_mean","macro_f1_sd",
    "balanced_accuracy_mean","positive_recall_mean",
    "roc_auc_mean","pr_auc_mean"
]].sort_values(["dataset","macro_f1_mean"],ascending=[True,False]).round(6))

## 3. Compact manuscript-ready performance table

In [ ]:
# A compact primary table: Macro-F1 plus key diagnostic metrics.
table=summary[[
    "model","dataset","macro_f1_mean","macro_f1_sd",
    "balanced_accuracy_mean","positive_recall_mean","roc_auc_mean","pr_auc_mean"
]].copy()

table["Macro-F1"]=table.apply(
    lambda r: f"{r.macro_f1_mean:.3f} ± {r.macro_f1_sd:.3f}"
              if pd.notna(r.macro_f1_sd) else f"{r.macro_f1_mean:.3f}",
    axis=1
)
table["Balanced Accuracy"]=table.balanced_accuracy_mean.map(lambda x:f"{x:.3f}")
table["Positive Recall"]=table.positive_recall_mean.map(lambda x:f"{x:.3f}")
table["ROC-AUC"]=table.roc_auc_mean.map(lambda x:f"{x:.3f}")
table["PR-AUC"]=table.pr_auc_mean.map(lambda x:f"{x:.3f}")

manuscript_table=table[[
    "dataset","model","Macro-F1","Balanced Accuracy","Positive Recall","ROC-AUC","PR-AUC"
]].sort_values(["dataset","model"])

display(manuscript_table)
manuscript_table.to_csv(OUT/"MANUSCRIPT_PRIMARY_PERFORMANCE_TABLE.csv",index=False)

## 4. Canonical decision summary

In [ ]:
decision=[]
classicals=["Logistic_Regression","Multinomial_NB","Linear_SVM","Random_Forest"]

for dname in datasets:
    sub=summary[summary.dataset==dname].sort_values("macro_f1_mean",ascending=False)
    best=sub.iloc[0]
    bc=sub[sub.model.isin(classicals)].sort_values("macro_f1_mean",ascending=False).iloc[0]

    decision.append({
        "dataset":dname,
        "best_model":best.model,
        "best_macro_f1":best.macro_f1_mean,
        "best_classical_model":bc.model,
        "best_classical_macro_f1":bc.macro_f1_mean,
        "roberta_macro_f1":float(sub[sub.model=="RoBERTa"].macro_f1_mean.iloc[0]),
        "distilbert_macro_f1":float(sub[sub.model=="DistilBERT"].macro_f1_mean.iloc[0])
    })

decision=pd.DataFrame(decision)
display(decision.round(6))
decision.to_csv(OUT/"FINAL_DECISION_SUMMARY.csv",index=False)

## 5. Combine the key diagnostics into one summary

In [ ]:
prior=pd.read_csv(BR/"roberta_label_shift_EM_sensitivity.csv")
prior_summary=(prior.groupby("dataset")
    .agg(
        original_macro_f1=("original_macro_f1","mean"),
        adjusted_macro_f1=("adjusted_macro_f1","mean"),
        label_shift_delta=("delta_macro_f1","mean"),
        estimated_target_prior=("estimated_target_prior_positive_unlabeled","mean"),
        actual_target_prior=("actual_target_prior_positive_for_evaluation_only","mean"),
        original_positive_recall=("original_positive_recall","mean"),
        adjusted_positive_recall=("adjusted_positive_recall","mean")
    ).reset_index())

coverage=pd.read_csv(BR/"source_tfidf_feature_coverage.csv")

length=pd.read_csv(BR/"length_control_dreaddit.csv")
length_summary=(length.groupby(["model","variant"])
    .agg(macro_f1=("macro_f1","mean"),positive_recall=("positive_recall","mean"),roc_auc=("roc_auc","mean"))
    .reset_index())
lw=length_summary.pivot(index="model",columns="variant",values="macro_f1")
lw["delta_macro_f1"]=lw["twitter_length_truncated"]-lw["original"]

print("RoBERTa label-shift sensitivity:")
display(prior_summary.round(6))
print("\nTF-IDF source-feature coverage:")
display(coverage.round(6))
print("\nLength-control Macro-F1:")
display(lw.round(6))

prior_summary.to_csv(OUT/"DIAGNOSTIC_label_shift_summary.csv",index=False)
coverage.to_csv(OUT/"DIAGNOSTIC_lexical_coverage.csv",index=False)
lw.reset_index().to_csv(OUT/"DIAGNOSTIC_length_control_summary.csv",index=False)

## 6. Revised Macro-F1 figure with reference baselines

In [ ]:
baselines=pd.read_csv(BR/"label_only_baselines_stageB.csv")

model_order=["Logistic_Regression","Multinomial_NB","Linear_SVM","Random_Forest","DistilBERT","RoBERTa"]
dataset_order=["Dreaddit_Test","Reddit_Title","Reddit_Combi","Twitter_Full","Twitter_NonAdvert"]
labels=["Dreaddit\nTest","Reddit\nTitle","Reddit\nCombi","Twitter\nFull","Twitter\nNonAdvert"]

fig,ax=plt.subplots(figsize=(11,6.2))
x=np.arange(len(dataset_order))
width=0.12

for i,m in enumerate(model_order):
    vals=[]
    errs=[]
    for d in dataset_order:
        r=summary[(summary.model==m)&(summary.dataset==d)].iloc[0]
        vals.append(r.macro_f1_mean)
        errs.append(0 if pd.isna(r.macro_f1_sd) else r.macro_f1_sd)
    ax.bar(x+(i-2.5)*width,vals,width,yerr=errs,capsize=2,label=m.replace("_"," "))

# source-prior random reference as black points
bmap=baselines.set_index("dataset")
rand=[bmap.loc[d,"source_prior_random_macro_f1_mean"] for d in dataset_order]
ax.scatter(x,rand,marker="x",s=55,label="Source-prior random baseline",zorder=5)

ax.set_xticks(x,labels)
ax.set_ylabel("Macro-F1")
ax.set_ylim(0.20,0.90)
ax.set_xlabel("Evaluation dataset")
ax.legend(ncol=2,frameon=False,fontsize=8.5)
ax.grid(axis="y",linestyle="--",linewidth=.5,alpha=.5)
fig.tight_layout()
fig.savefig(OUT/"Figure_MacroF1_AllSixModels_Canonical.png",dpi=320,bbox_inches="tight")
plt.show()

## Final rule for manuscript revision

Use only files inside:

`MyDrive/stress_detection_reviewer_revision/final_canonical`

for the revised performance tables.

- Stage A2 MNB is canonical.
- Stage B non-MNB results remain valid.
- Do not use the stale Stage-B MNB rows.